# 1. Thu thập, chuẩn bị và mô tả dữ liệu

Notebook cốt lõi cho Chương 1 và Chương 6 của học phần.

**Mục tiêu:** giới thiệu dữ liệu, kiểm tra chất lượng, phân loại biến và mô tả `G3`.
Các phân tích khám phá chi tiết hơn nằm trong `notebooks/appendix/01_EDA.ipynb`.

In [1]:
from pathlib import Path
import warnings

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
ALPHA = 0.05
ROOT = Path("../..")
DATA_RAW = ROOT / "data" / "raw"
DATA_OUT = ROOT / "data" / "processed"
FIGURES = ROOT / "report" / "figures"
DATA_OUT.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

## 1.1. Nguồn dữ liệu và đơn vị quan sát

In [ ]:
df = pd.read_csv(DATA_RAW / "student-mat.csv", sep=";")
print(f"Kích thước dữ liệu: {df.shape[0]} học sinh, {df.shape[1]} biến")
print(f"Giá trị thiếu: {int(df.isna().sum().sum())}")
print(f"Dòng trùng lặp: {int(df.duplicated().sum())}")
print("Khoảng G3:", (df["G3"].min(), df["G3"].max()))
df.head()

Mỗi dòng là một học sinh môn Toán tại một trong hai trường. Đây là dữ liệu quan sát,
không phải dữ liệu từ thí nghiệm ngẫu nhiên. Vì vậy kết quả mô tả mối liên hệ, không tự
động chứng minh quan hệ nhân quả.

## 1.2. Phân loại biến

In [ ]:
nominal = ["school", "sex", "address", "famsize", "Pstatus", "Mjob", "Fjob",
           "reason", "guardian", "schoolsup", "famsup", "paid", "activities",
           "nursery", "higher", "internet", "romantic"]
ordinal = ["Medu", "Fedu", "traveltime", "studytime", "failures", "famrel",
           "freetime", "goout", "Dalc", "Walc", "health"]
count = ["age", "absences"]
score = ["G1", "G2", "G3"]

variable_types = pd.DataFrame({
    "loai_bien": ["Định danh", "Thứ bậc", "Đếm", "Điểm số"],
    "so_bien": [len(nominal), len(ordinal), len(count), len(score)],
    "vi_du": [", ".join(nominal[:4]), ", ".join(ordinal[:4]), ", ".join(count), ", ".join(score)],
})
variable_types

## 1.3. Thống kê mô tả và biểu diễn

In [ ]:
summary = df[["age", "absences", "G1", "G2", "G3"]].describe().T
summary

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df["G3"], bins=range(0, 22), discrete=True, ax=axes[0], color="#4C78A8")
axes[0].axvline(df["G3"].mean(), color="black", linestyle="--", label=f"Mean={df['G3'].mean():.2f}")
axes[0].set_title("Phân phối điểm cuối kỳ G3")
axes[0].legend()
sns.boxplot(data=df, x="school", y="G3", ax=axes[1], color="#72B7B2")
axes[1].set_title("G3 theo trường")
fig.tight_layout()
fig.savefig(FIGURES / "eda_course_overview.png", dpi=150, bbox_inches="tight")
plt.show()

## 1.4. Chuẩn bị dữ liệu phân tích

In [ ]:
# Không sửa dữ liệu raw. G3=0 và các giá trị absences lớn vẫn được giữ.
df.to_csv(DATA_OUT / "student_mat_clean.csv", index=False)
print("Đã xuất dữ liệu phân tích:", DATA_OUT / "student_mat_clean.csv")

## Kết luận

- Dữ liệu có 395 học sinh và 33 biến, không có missing hoặc dòng trùng lặp.
- `G3` nằm trên thang 0-20; trung bình khoảng 10,42 và trung vị 11.
- Các biến gồm định danh, thứ bậc, biến đếm và điểm số; loại thang đo cần được xét khi
  chọn phương pháp phân tích.
- Mẫu chỉ đến từ hai trường, nên khả năng khái quát hóa còn hạn chế.